# Chain of Thought - 第三部分：Auto-CoT 与高级技巧

## 学习目标
1. 掌握 Auto-CoT 自动生成示例
2. 学习 CoT 提示构建器
3. 了解高级优化技巧

## 目录
1. [Auto-CoT 原理](#1-auto-cot-原理)
2. [AutoCoT 类使用](#2-autocot-类使用)
3. [CoTPromptBuilder](#3-cotpromptbuilder)
4. [高级技巧](#4-高级技巧)
5. [最佳实践总结](#5-最佳实践总结)
6. [练习](#6-练习)

In [ ]:
import sys
sys.path.insert(0, '..')

from src.chain_of_thought import (
    CoTExample, CoTResult, CoTPromptBuilder, AutoCoT, FewShotCoT
)
print("模块加载成功！")

---
## 1. Auto-CoT 原理

### 1.1 核心思想

Auto-CoT (Zhang et al., 2022) 自动生成多样化的推理示例，无需人工标注。

In [ ]:
print("""
Auto-CoT 工作流程：

┌─────────────────────────────────────────────────────┐
│  1. 问题聚类                                         │
│     将问题集按相似性分成K个簇                         │
│                                                     │
│  2. 代表性选择                                       │
│     从每个簇选择一个代表性问题                        │
│                                                     │
│  3. 推理链生成                                       │
│     用 Zero-shot CoT 为每个代表生成推理链             │
│                                                     │
│  4. 组合示例                                         │
│     将生成的示例组合成 Few-shot 提示                  │
└─────────────────────────────────────────────────────┘

优势：
  - 无需人工标注
  - 示例多样性好
  - 可扩展到大规模数据
""")

### 1.2 与其他方法对比

In [ ]:
print("""
┌──────────────┬─────────────┬─────────────┬─────────────┐
│     特性     │  Zero-shot  │  Few-shot   │  Auto-CoT   │
├──────────────┼─────────────┼─────────────┼─────────────┤
│ 人工标注     │ 无          │ 需要        │ 无          │
│ 示例质量     │ N/A         │ 高(人工)    │ 中(自动)    │
│ 多样性       │ N/A         │ 依赖设计    │ 自动保证    │
│ 可扩展性     │ 高          │ 低          │ 高          │
│ 效果         │ 基础        │ 最好        │ 较好        │
└──────────────┴─────────────┴─────────────┴─────────────┘
""")

---
## 2. AutoCoT 类使用

### 2.1 基本使用

In [ ]:
# 创建 AutoCoT 实例
auto_cot = AutoCoT(n_demonstrations=3)

print(f"AutoCoT 配置：")
print(f"  示例数: {auto_cot._n_demonstrations}")

In [ ]:
# 准备问题集
questions = [
    "5 + 3 = ?",
    "12 - 7 = ?",
    "8 × 4 = ?",
    "20 ÷ 5 = ?",
    "15 + 8 - 3 = ?",
    "6 × 7 + 2 = ?",
]

print(f"问题集 ({len(questions)} 个问题)：")
for i, q in enumerate(questions, 1):
    print(f"  {i}. {q}")

In [ ]:
# 生成示例（无LLM时返回模拟结果）
demos = auto_cot.generate_demonstrations(questions)

print(f"\n生成的示例 ({len(demos)} 个)：")
for i, ex in enumerate(demos, 1):
    print(f"\n示例 {i}:")
    print(f"  问题: {ex.question}")
    print(f"  推理: {ex.reasoning[:50]}...")

### 2.2 聚类策略

In [ ]:
print("""
Auto-CoT 聚类策略：

1. 简单聚类（默认）
   - 基于问题长度和关键词
   - 快速但不够精确

2. 嵌入聚类
   - 使用句子嵌入
   - 语义相似性更准确
   - 需要嵌入模型

3. 混合聚类
   - 结合多种特征
   - 效果最好但最慢
""")

---
## 3. CoTPromptBuilder

### 3.1 构建器模式

In [ ]:
# 使用构建器创建提示
builder = CoTPromptBuilder()

# 链式调用
prompt = (
    builder
    .set_question("计算 25 × 4 的结果")
    .build()
)

print("基础提示：")
print(prompt)

In [ ]:
# 添加示例
example = CoTExample(
    question="计算 12 × 3",
    reasoning="12 × 3 = 36",
    answer="36"
)

prompt_with_example = (
    CoTPromptBuilder()
    .add_example(example)
    .set_question("计算 25 × 4")
    .build()
)

print("带示例的提示：")
print(prompt_with_example)

### 3.2 自定义系统提示

In [ ]:
# 自定义提示 - 使用 set_instruction
custom_prompt = (
    CoTPromptBuilder()
    .set_instruction("你是一个数学老师，请详细解释每一步。")
    .set_question("解方程 2x + 5 = 15")
    .build()
)

print("自定义提示：")
print(custom_prompt)

---
## 4. 高级技巧

### 4.1 复杂度自适应

In [ ]:
def adaptive_cot(question: str, complexity: str = "auto"):
    """根据问题复杂度选择策略"""
    
    # 简单启发式判断复杂度
    if complexity == "auto":
        if len(question) < 20:
            complexity = "simple"
        elif any(w in question for w in ["如果", "假设", "证明"]):
            complexity = "complex"
        else:
            complexity = "medium"
    
    strategies = {
        "simple": "Zero-shot CoT",
        "medium": "Few-shot CoT (3示例)",
        "complex": "Few-shot CoT (5示例) + 自一致性"
    }
    
    return strategies.get(complexity, "Few-shot CoT")

# 测试
test_qs = [
    "5+3=?",
    "商店有50个苹果，卖出20个，还剩多少？",
    "如果所有A都是B，所有B都是C，那么所有A都是C吗？证明你的答案。"
]

print("复杂度自适应：")
for q in test_qs:
    strategy = adaptive_cot(q)
    print(f"  问题: {q[:30]}...")
    print(f"  策略: {strategy}\n")

### 4.2 推理链验证

In [ ]:
def validate_reasoning(reasoning: str) -> dict:
    """验证推理链质量"""
    issues = []
    
    # 检查步骤数
    lines = [l for l in reasoning.split('\n') if l.strip()]
    if len(lines) < 2:
        issues.append("步骤太少")
    
    # 检查是否有数字/计算
    import re
    if not re.search(r'\d', reasoning):
        issues.append("缺少具体数值")
    
    # 检查是否有等号（计算过程）
    if '=' not in reasoning:
        issues.append("缺少计算过程")
    
    return {
        "valid": len(issues) == 0,
        "issues": issues,
        "step_count": len(lines)
    }

# 测试
good_reasoning = """1. 初始：50个
2. 卖出：50-20=30个
3. 进货：30+10=40个"""

bad_reasoning = "答案是40个"

print("推理链验证：")
print(f"好的推理: {validate_reasoning(good_reasoning)}")
print(f"差的推理: {validate_reasoning(bad_reasoning)}")

### 4.3 错误恢复

In [ ]:
print("""
CoT 错误恢复策略：

1. 重试策略
   - 检测到错误时重新生成
   - 使用不同温度参数

2. 自一致性
   - 多次采样取多数答案
   - 过滤异常推理链

3. 验证反馈
   - 将验证结果反馈给模型
   - 要求修正错误步骤

4. 分步验证
   - 每步生成后验证
   - 错误时回退重试
""")

---
## 5. 最佳实践总结

In [ ]:
print("""
CoT 最佳实践：

1. 选择合适的策略
   - 简单任务: Zero-shot
   - 重要任务: Few-shot
   - 大规模: Auto-CoT

2. 示例设计
   - 步骤清晰，格式一致
   - 覆盖多种问题类型
   - 3-8个示例最佳

3. 质量保证
   - 验证推理链完整性
   - 使用自一致性提高可靠性
   - 监控和记录失败案例

4. 性能优化
   - 缓存常用示例
   - 按复杂度分层处理
   - 批量处理相似问题
""")

---
## 6. 练习

### 练习1：使用 AutoCoT

In [ ]:
# TODO: 创建一个问题集并使用 AutoCoT 生成示例
# my_questions = [...]
# my_auto_cot = AutoCoT(n_demonstrations=2)
# my_examples = my_auto_cot.generate_demonstrations(my_questions)

### 练习2：使用 CoTPromptBuilder

In [ ]:
# 自定义提示 - 使用 set_instruction
custom_prompt = (
    CoTPromptBuilder()
    .set_instruction("你是一个数学老师，请详细解释每一步。")
    .set_question("解方程 2x + 5 = 15")
    .build()
)

print("自定义提示：")
print(custom_prompt)

### 练习3：实现推理链验证

In [ ]:
# TODO: 扩展 validate_reasoning 函数，添加更多检查
# def my_validate_reasoning(reasoning: str) -> dict:
#     pass

---
## 总结

本系列教程覆盖了 Chain of Thought 的完整内容：

- **Part 1**: 基础概念和 Zero-shot CoT
- **Part 2**: Few-shot CoT 和示例设计
- **Part 3**: Auto-CoT 和高级技巧

下一步学习 **02_ReAct_tutorial.ipynb** 了解 ReAct 框架